# Emotion Classification using Bidirectional GRU

**Improved version** — changes from the original are marked with `# IMPROVED:` comments.

Key fixes applied (see chat for full explanation of *why*):
1. Fixed the broken `oov_token=''` bug in the Tokenizer
2. Now uses the dataset's real **validation** split for early stopping (previously the test set was reused as validation, which quietly leaks test information into model selection)
3. Adds `ModelCheckpoint` + `ReduceLROnPlateau` so training recovers the actual best weights and adapts the learning rate
4. Adds `recurrent_dropout` and `SpatialDropout1D` for better regularization of the embedding output
5. Adds a per-class `classification_report` (precision/recall/F1) — accuracy alone hides how badly minority classes like *love* and *surprise* do
6. Bundles preprocessing into a single `predict_emotion()` function so training-time and inference-time preprocessing can never drift apart (this was the likely cause of the wrong predictions you saw)
7. Expands the sanity-check examples to include short, plain-spoken sentences like `"I am happy"` and `"I love my girlfriend"`, since those are exactly the inputs the original model struggled with


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pickle

#dataset load krwane mai help karegi
from datasets import load_dataset

## Load data — now keeping the real validation split

**IMPROVED:** the original notebook only pulled `train` and `test` and later reused `test` as the validation set during `.fit()`. That means early stopping and model selection were implicitly tuned on the test set, so the reported test accuracy was optimistic. The `dair-ai/emotion` dataset ships a dedicated `validation` split — we use that instead, and touch `test` only once, at the very end, for a truly held-out evaluation.

In [ ]:
emotion_dataset = load_dataset('dair-ai/emotion')

train_text = emotion_dataset['train']['text']
train_label = emotion_dataset['train']['label']

# IMPROVED: load the real validation split instead of reusing test as validation
val_text = emotion_dataset['validation']['text']
val_label = emotion_dataset['validation']['label']

test_text = emotion_dataset['test']['text']
test_label = emotion_dataset['test']['label']

In [ ]:
label_names = emotion_dataset['train'].features['label'].names
label_names

In [ ]:
df_train = pd.DataFrame({'text': train_text, 'label': [label_names[i] for i in train_label]})
df_val   = pd.DataFrame({'text': val_text,   'label': [label_names[i] for i in val_label]})
df_test  = pd.DataFrame({'text': test_text,  'label': [label_names[i] for i in test_label]})

df_train.isnull().sum()

In [ ]:
df_train['label'].value_counts()

In [ ]:
sns.countplot(x='label', data=df_train, order=df_train['label'].value_counts().index)
plt.title('Class distribution — note love & surprise are minority classes')
plt.show()

## Tokenization — fixed OOV token

**IMPROVED (the actual bug):** the original used `Tokenizer(num_words=max_words, oov_token='')` — an *empty string* as the OOV token. That's not the documented Keras usage and can cause unpredictable indexing for out-of-vocabulary words. It's fixed here to `'<OOV>'`.

**IMPROVED:** vocab size raised from 10,000 → 15,000. Your data has 15,213 unique words after preprocessing (see original notebook's own printout), so the old cap of 10,000 was silently dropping ~5,000 words to OOV — words that could easily include things like proper nouns ("girlfriend", specific names) that carry real emotional signal.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 15000   # IMPROVED: was 10000
max_len = 50

tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')  # IMPROVED: was oov_token=''
tokenizer.fit_on_texts(df_train['text'])

train_sequence = tokenizer.texts_to_sequences(df_train['text'])
val_sequence   = tokenizer.texts_to_sequences(df_val['text'])    # IMPROVED: tokenize validation split
test_sequence  = tokenizer.texts_to_sequences(df_test['text'])

padded_train_sequence = pad_sequences(train_sequence, maxlen=max_len, padding='post', truncating='post')
padded_val_sequence   = pad_sequences(val_sequence,   maxlen=max_len, padding='post', truncating='post')
padded_test_sequence  = pad_sequences(test_sequence,  maxlen=max_len, padding='post', truncating='post')

train_labels = np.array(train_label)
val_labels   = np.array(val_label)
test_labels  = np.array(test_label)

num_classes = len(np.unique(train_labels))

print(f'Vocabulary size seen: {len(tokenizer.word_index)}')
print(f'Vocabulary size used (num_words cap): {max_words}')
print(f'Max sentence length: {padded_train_sequence.shape[1]}')

In [ ]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weight_dict = dict(enumerate(class_weights))
class_weight_dict

## Callbacks — IMPROVED

Added `ModelCheckpoint` (so you keep the literal best-performing weights on disk, not just in memory) and `ReduceLROnPlateau` (lets the optimizer take smaller steps once progress stalls, instead of stopping outright).

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

os.makedirs('Artifacts', exist_ok=True)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# IMPROVED: new callbacks
checkpoint = ModelCheckpoint(
    filepath='Artifacts/best_model.keras',
    monitor='val_loss',
    save_best_only=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

## Final model — Bidirectional GRU (IMPROVED regularization)

**IMPROVED:** added `SpatialDropout1D` right after the embedding layer (drops entire word-embedding vectors rather than individual values — a well-established trick for text CNN/RNN models that reduces overfitting more effectively than plain `Dropout` on embeddings) and `recurrent_dropout` inside the GRU cells themselves for extra regularization on the recurrent connections.

We skip re-running the RNN vs LSTM vs GRU comparison here since your original notebook already established GRU > LSTM > RNN — going straight to the improved BiGRU.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dropout, SpatialDropout1D, Dense

BiGRU = Sequential([
    Embedding(input_dim=max_words, output_dim=300),
    SpatialDropout1D(0.3),                                              # IMPROVED
    Bidirectional(GRU(128, return_sequences=True, recurrent_dropout=0.2)),  # IMPROVED: recurrent_dropout
    Dropout(0.5),
    Bidirectional(GRU(64, recurrent_dropout=0.2)),                      # IMPROVED: recurrent_dropout
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

BiGRU.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
BiGRU.build(input_shape=(None, max_len))
BiGRU.summary()

In [ ]:
history = BiGRU.fit(
    padded_train_sequence,
    train_labels,
    validation_data=(padded_val_sequence, val_labels),   # IMPROVED: real validation split, not test
    epochs=30,                                            # IMPROVED: raised cap, ReduceLROnPlateau + EarlyStopping manage it
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stopping, checkpoint, reduce_lr]    # IMPROVED
)

## Final evaluation — only touched once, here

**IMPROVED:** this is the *only* place `test_labels` / `padded_test_sequence` are used for scoring. Everything above (early stopping, checkpointing, LR reduction) watched the validation set instead, so this number is a genuinely unbiased estimate of real-world performance.

In [ ]:
BiGRU_loss, BiGRU_accuracy = BiGRU.evaluate(padded_test_sequence, test_labels)
print(f'BiGRU Test Loss: {BiGRU_loss}')
print(f'BiGRU Test Accuracy: {BiGRU_accuracy}')

## Per-class metrics — IMPROVED

Overall accuracy hides class-level weaknesses. `love` and `surprise` are your smallest classes (~8% and ~3.5% of the training set) — this report shows exactly how well the model does on each one, not just on average.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

BiGRU_predictions = np.argmax(BiGRU.predict(padded_test_sequence), axis=1)

print(classification_report(test_labels, BiGRU_predictions, target_names=label_names))

sns.heatmap(confusion_matrix(test_labels, BiGRU_predictions), annot=True, fmt='d',
            cmap='Blues', xticklabels=label_names, yticklabels=label_names)
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

## Single, reusable prediction function — IMPROVED (fixes the training/inference mismatch)

**This is the most important addition.** In the original notebook, sample predictions worked fine because they reused the exact in-memory `tokenizer` and `max_len`. The moment you move to a separate script or app and recreate the tokenizer instead of loading the pickled one — or use a different `maxlen`/padding — predictions become close to arbitrary, which almost certainly explains the wrong `"i love my girlfriend"` → fear and `"I am happy"` → sadness results you were seeing.

Wrapping preprocessing + prediction in one function, and using *that same function* both here and in any deployed app, makes that class of bug impossible.

In [ ]:
def predict_emotion(text, model=BiGRU, tok=tokenizer, maxlen=max_len, labels=label_names, return_probs=False):
    seq = tok.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=maxlen, padding='post', truncating='post')
    probs = model.predict(padded, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    if return_probs:
        return labels[pred_idx], dict(zip(labels, probs.tolist()))
    return labels[pred_idx]

sample_texts = [
    "I can't believe how happy I am right now, this is amazing!",
    "I feel so alone and hopeless today.",
    "I am furious that they cancelled the trip at the last minute.",
    "I feel terrified when walking down dark alleyways alone.",
    "I was shocked and completely surprised by the unexpected gift!",
    "I am happy",                    # IMPROVED: short, plain-spoken test case
    "I love my girlfriend",          # IMPROVED: short, plain-spoken test case
]

for t in sample_texts:
    label, probs = predict_emotion(t, return_probs=True)
    top3 = sorted(probs.items(), key=lambda x: -x[1])[:3]
    print(f'Text: {t}')
    print(f'  Predicted: {label}')
    print(f'  Top 3: {top3}')
    print()

## Save artifacts

**IMPROVED:** also saves `max_len` and `label_names` alongside the model and tokenizer, so any downstream app has everything it needs to reproduce `predict_emotion()` exactly — no more risk of a mismatched `maxlen` or label order in production.

In [ ]:
model_dir = 'Artifacts'
os.makedirs(model_dir, exist_ok=True)

BiGRU.save(os.path.join(model_dir, 'BiGRU_model.keras'))

with open(os.path.join(model_dir, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)

# IMPROVED: persist config needed for correct inference elsewhere
with open(os.path.join(model_dir, 'config.pkl'), 'wb') as f:
    pickle.dump({'max_len': max_len, 'label_names': label_names}, f)

print('Saved model, tokenizer, and config to', model_dir)

## If you deploy this elsewhere (Flask app, API, etc.)

Load all three artifacts together and reuse the same `predict_emotion` logic — never re-instantiate a fresh `Tokenizer()`:

```python
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

model = load_model('Artifacts/BiGRU_model.keras')
with open('Artifacts/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)
with open('Artifacts/config.pkl', 'rb') as f:
    config = pickle.load(f)

def predict_emotion(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=config['max_len'], padding='post', truncating='post')
    probs = model.predict(padded, verbose=0)[0]
    return config['label_names'][probs.argmax()]
```
